# Milestone 4 — train the landmark model

**Inputs to attach:** the crop cache (Add Input → Notebooks → the milestone-2 notebook, or the cache dataset). `cache.dir` in `configs/layer1_base.yaml` must match the mount path.
**Accelerator:** GPU (T4). With the RAM cache a full run is well under an hour; if an epoch takes more than ~30 s, the dataloader is the problem — profile before touching anything else.

The notebook is thin on purpose: clone, install, `python train.py`. Checkpoints land in `/kaggle/working/checkpoints` every epoch, metrics append to `/kaggle/working/metrics.csv`, and curves render to `/kaggle/working/curves.png` — so a dead session costs nothing but compute time.

**Session survival:** to continue an interrupted run in a fresh session, attach the previous version's output (so `checkpoints/last.pth` is reachable), copy it into `/kaggle/working/checkpoints/`, and run with `--resume`. The resume path restores optimiser, scheduler, epoch counter and RNG state — verified by `tests/test_resume.py` to reproduce an uninterrupted run exactly. `train.stop_after_epochs` gives a clean stop before the 12 h cap.

**Loss comparison:** run once with `train.loss: l2`, then again with `wing` (edit the config or duplicate it); every run snapshots its config next to the checkpoints.

In [ ]:
!rm -rf dms-layer1
!git clone -b claude/facial-landmark-perception-q80yhn https://github.com/keerthanpragnay1728-prog/dms-layer1.git
%cd dms-layer1
# provenance: confirm the commit this run uses BEFORE trusting any number
!git log --oneline -1
!pip install -q -r requirements.txt

In [ ]:
!python tests/run_tests.py

In [ ]:
!python train.py --config configs/layer1_base.yaml
# interrupted earlier? attach the old output, restore checkpoints/, then:
# !python train.py --config configs/layer1_base.yaml --resume

In [ ]:
import csv
from IPython.display import Image, display
with open('/kaggle/working/metrics.csv') as f:
    rows = list(csv.DictReader(f))
best = min(rows, key=lambda r: float(r['val_nme_pct']))
print(f"{len(rows)} epochs; best val NME {best['val_nme_pct']}% at epoch {best['epoch']}")
print('median epoch seconds:', sorted(float(r['seconds']) for r in rows)[len(rows)//2])
display(Image(filename='/kaggle/working/curves.png'))